In [10]:
# Cell 1: Environment Check
print("🔧 Environment Setup for Club Piscine MMM")
print("=" * 50)

# Check Python version
import sys
print(f"Python version: {sys.version}")

# Verify package installations
packages = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'yaml', 'openmeteo_requests', 'statsmodels', 'openpyxl']
missing = []

for pkg in packages:
    try:
        __import__(pkg.replace('-', '_'))
        print(f"✓ {pkg} is installed")
    except ImportError:
        print(f"✗ {pkg} is NOT installed")
        missing.append(pkg)

if missing:
    print(f"\n⚠️ Missing packages: {missing}")
    print("Installing missing packages...")
    import subprocess
    for pkg in missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
else:
    print("\n✅ All required packages are installed!")

🔧 Environment Setup for Club Piscine MMM
Python version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 11:09:21) [Clang 14.0.6 ]
✓ pandas is installed
✓ numpy is installed
✓ matplotlib is installed
✓ seaborn is installed
✓ yaml is installed
✓ openmeteo_requests is installed
✓ statsmodels is installed
✓ openpyxl is installed

✅ All required packages are installed!


In [11]:
# Import required packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import statsmodels.api as sm
import openpyxl

In [12]:
# Cell 2: Project Structure Verification
import os
from pathlib import Path

# Define project root
project_root = Path().cwd().parent if Path().cwd().name == 'notebooks' else Path().cwd()
print(f"Project root: {project_root}")

# Check if all required directories exist
required_dirs = [
    'data/raw',
    'data/interim', 
    'data/processed',
    'notebooks',
    'config',
    'src/data',
    'src/features',
    'src/models',
    'src/analysis',
    'reports/figures'
]

print("\n Checking project structure:")
all_good = True
for dir_path in required_dirs:
    full_path = project_root / dir_path
    if full_path.exists():
        print(f"✓ {dir_path}/")
    else:
        print(f"✗ {dir_path}/ (MISSING)")
        all_good = False

if all_good:
    print("\n Project structure is complete")
else:
    print("\n Some directories are missing. Creating them...")
    for dir_path in required_dirs:
        full_path = project_root / dir_path
        full_path.mkdir(parents=True, exist_ok=True)
        print(f"Created: {full_path}")

Project root: /Users/raoul/Dev/busa693-clubpiscine/club-piscine-mmm-projectfolder

 Checking project structure:
✓ data/raw/
✓ data/interim/
✓ data/processed/
✓ notebooks/
✓ config/
✓ src/data/
✓ src/features/
✓ src/models/
✓ src/analysis/
✓ reports/figures/

 Project structure is complete


In [13]:
#  List of Raw Data Files
raw_path = project_root / 'data' / 'raw'
print(f" Looking for data files in: {raw_path}")

if raw_path.exists():
    files = list(raw_path.glob('*'))
    if files:
        print(f"Found {len(files)} file(s):")
        for i, file_path in enumerate(files, 1):
            size_kb = file_path.stat().st_size / 1024
            print(f"{i:2}. {file_path.name:40} ({size_kb:.1f} KB)")
            
        # Identifying file types
        print("\n Trying to identify file types:")
        for file_path in files:
            if file_path.suffix.lower() in ['.csv', '.xlsx', '.xls']:
                print(f"\n{file_path.name}:")
                try:
                    if file_path.suffix.lower() == '.csv':
                        # Try different encodings
                        encodings = ['utf-8', 'latin-1', 'ISO-8859-1', 'cp1252']
                        df = None
                        for enc in encodings:
                            try:
                                df = pd.read_csv(file_path, encoding=enc, nrows=5)
                                print(f"  ✓ Can read with {enc} encoding")
                                break
                            except:
                                continue
                        if df is not None:
                            print(f"  Columns: {list(df.columns)}")
                            print(f"  Shape preview: {df.shape}")
                    elif file_path.suffix.lower() in ['.xlsx', '.xls']:
                        df = pd.read_excel(file_path, nrows=5)
                        print(f"  ✓ Can read Excel file")
                        print(f"  Columns: {list(df.columns)}")
                        print(f"  Shape preview: {df.shape}")
                except Exception as e:
                    print(f"  ✗ Error reading: {str(e)[:100]}...")
    else:
        print("No files found in data/raw/")
        print("Please add files to this directory.")
else:
    print("data/raw/ directory doesn't exist!")

 Looking for data files in: /Users/raoul/Dev/busa693-clubpiscine/club-piscine-mmm-projectfolder/data/raw
Found 6 file(s):
 1. CalendrierFiscal.xlsx                    (349.7 KB)
 2. Budget 2025 - 21 août.xlsx               (67.6 KB)
 3. Rapport de soumissions 2024.xlsx         (3169.6 KB)
 4. +Rapport de soumissions 2025.xlsx        (2117.9 KB)
 5. Recap_Tableau_Medias_2025.xlsx           (200.4 KB)
 6. Budget 2024 - REEL au 5 novembre.xlsx    (103.9 KB)

 Trying to identify file types:

CalendrierFiscal.xlsx:
  ✓ Can read Excel file
  Columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2']
  Shape preview: (5, 3)

Budget 2025 - 21 août.xlsx:
  ✓ Can read Excel file
  Columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'VERSION', 'BUDGET 2025 - V6', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: